In [24]:
from sympy import Function, Symbol, symbols, simplify, Eq, Symbol, latex, pprint, collect, expand
from sympy import init_printing
from IPython.display import display, Math

init_printing(use_latex=True)

In [25]:
import re
from IPython.display import display, Math

def color_terms(latex_str):
    # Color all q_i(t) terms orange
    latex_str = re.sub(
        r'(q_\{(\d+)\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{orange} \1{\\left(\3\\right)}}',
        latex_str
    )
    latex_str = re.sub(
        r'(\\epsilon_\{([A])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{Cyan} \1{\\left(\3\\right)}}',
        latex_str
    )
    latex_str = re.sub(
        r'(\\epsilon_\{([B])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{Aquamarine} \1{\\left(\3\\right)}}',
        latex_str
    )
    latex_str = re.sub(
        r'(\\epsilon_\{([C])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{SpringGreen} \1{\\left(\3\\right)}}',
        latex_str
    )
    return latex_str

In [26]:
# ── time variable ─────────────────────────────────────────────────────────────
t = Symbol('t')

# ── delay parameters ──────────────────────────────────────────────────────────
tau12, tau21, tau13, tau31, tau23, tau32 = symbols(
    r'\tau_{12} \tau_{21} \tau_{13} \tau_{31} \tau_{23} \tau_{32}',
    real=True, positive=True
)

# ── laser angular frequencies (physical + measurement offsets) ────────────────
omega1, omega2, omega3 = symbols(r'\omega_1 \omega_2 \omega_3', real=True)
omega1m, omega2m, omega3m = symbols(r'\omega_1^m \omega_2^m \omega_3^m', real=True)

For now, we assume frequencies to be constant in time. Hartwig et. al. did a bit of analysis how this affects the clock noise removal.

In [27]:
# ── abstract time-dependent functions ─────────────────────────────────────────
phi1  = Function(r'\phi_1')   # laser phase noise, s/c 1
phi2  = Function(r'\phi_2')
phi3  = Function(r'\phi_3')

epsilonA = Function(r'\epsilon_A')  # clock noise, s/c 1 (master)
epsilonB = Function(r'\epsilon_B')  # clock noise, s/c 2
epsilonC = Function(r'\epsilon_C')  # clock noise, s/c 3

omegaREFA, omegaREFB, omegaREFC = symbols(r'\omega^{REF}_A \omega^{REF}_B \omega^{REF}_C', real=True)

q1 = Function('q_1')
q2 = Function('q_2')
q3 = Function('q_3')

In [28]:
def D(expr, tau):
    return expr.subs(t, t - tau)

In [58]:
# Map spacecraft index → (phi, q, omega, omega_m, clock_epsilon)
sc = {
    1: (phi1, q1, omega1, omega1m, epsilonA, omegaREFA),
    2: (phi2, q2, omega2, omega2m, epsilonB, omegaREFB),
    3: (phi3, q3, omega3, omega3m, epsilonC, omegaREFC),
}

tau = {
    (1,2): tau12, (2,1): tau21,
    (1,3): tau13, (3,1): tau31,
    (2,3): tau23, (3,2): tau32,
}

eta    = {}
etaSB  = {}
etaU  = {} # unmixed
REF  = {}
r = {}

include_phi = False 
include_clock_noise = True
include_board_jitter = True  

for (i, j) in tau:
    phi_i, q_i, om_i, omm_i, eps_i, omegaREFi = sc[i]
    phi_j, q_j, om_j, omm_j, eps_j, omegaREFj = sc[j]
    t_ij = tau[(i, j)]

    phi_terms = (D(phi_j(t), t_ij) - phi_i(t)) if include_phi else 0

    clock_terms_C = (- (om_j - om_i) * q_i(t)) if include_clock_noise else 0
    clock_terms_SB = (- (om_j - om_i + omm_j - omm_i) * q_i(t) - omm_i * q_i(t) + omm_j * D(q_j(t), t_ij)) if include_clock_noise else 0
    
    board_terms_c = (om_j * (eps_i(t) - D(eps_i(t), t_ij))) if include_board_jitter else 0
    board_terms_SB = ((om_j + omm_j) * (eps_i(t) - D(eps_i(t), t_ij))) if include_board_jitter else 0

    eta[(i,j)] = (phi_terms + clock_terms_C + board_terms_c)

    etaSB[(i,j)] = collect(expand(phi_terms + clock_terms_SB + board_terms_SB), [q_i(t), q_j(t)])

    r[(i,j)] = simplify((eta[(i,j)] - etaSB[(i,j)]) / omm_j)

    etaU[(i,j)] = (D(phi_j(t), t_ij) - om_j  * q_i(t) + (om_j * (eps_i(t) - D(eps_i(t), t_ij))))
    
    REF[(i,j)] = ( - int(include_clock_noise)*q_j(t) + int(include_board_jitter) * eps_i(t) ) # REF measured by the other PM, missing omegaREFi  *

The $\epsilon(t)$ terms originate from clock noise on the delay line boards which we will use three of. So we have A, B and C. Each corresponds to one spacecraft so the board A is where we apply delays 2 $\rightarrow$ 1 and 3 $\rightarrow$ 1. The $q(t)$ terms come from the Mokus. These are LISA-like, as in LISA will see similat clock noise coupling from the PM. All clock terms are given w.r.t the global clock, which we won't really ever know.

In [59]:
if True:
    for (i, j) in tau:
        display(Math(r'\eta_{' + str(i) + str(j) + '} = ' + color_terms(latex(eta[(i,j)]))))
        #display(Math(r'\eta_{' + str(i) + str(j) + '}^{U} = ' + color_terms(latex(etaU[(i,j)]))))
        display(Math(r'\eta_{' + str(i) + str(j) + '}^{SB} = ' + color_terms(latex(etaSB[(i,j)]))))
        display(Math(r'\frac{r_{' + str(i) + str(j) + r'}}{\omega_{' + str(j) + '}^m} = ' + color_terms(latex(r[(i,j)]))))
        display(Math(r'\frac{REF_{' + str(i) + str(j) + r'}}{ \omega_{' + str(j) + '}^m} = ' + color_terms(latex(REF[(i,j)]))))
        print("------------------------------------------------------")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

------------------------------------------------------


...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

------------------------------------------------------


We actually look at first generation TDI here. I do not know how to properly cancel out the timechaning delays with symbolic calculations.

In [48]:
def P12(expr): return expr - D(expr, tau13 + tau31)
def P21(expr): return D(expr, tau12) - D(expr, tau12 + tau13 + tau31)
def P13(expr): return -(expr - D(expr, tau12 + tau21))
def P31(expr): return -(D(expr, tau13) - D(expr, tau13 + tau12 + tau21))

# X1
X1 = collect(expand(P13(eta[(1,3)] + D(eta[(3,1)], tau13))+ P12(eta[(1,2)] + D(eta[(2,1)], tau12))), [q1(t), q2(t), q3(t), epsilonA(t), epsilonB(t), epsilonC(t)])

In [61]:
display(Math(r'X_1 = ' + color_terms(latex(X1)))) 

<IPython.core.display.Math object>

In [50]:
R = {}

R[(1,2)] = -(r[(1,3)] + D(r[(3,1)], tau13))

R[(1,3)] = r[(1,2)] + D(r[(2,1)], tau12)

R[(2,1)] = (r[(1,2)]
             - r[(1,3)]
             - D(r[(3,1)], tau13)
             - D(r[(1,2)], tau13 + tau31))

R[(3,1)] = (-r[(1,3)]
              + r[(1,2)]
              + D(r[(2,1)], tau12)
              + D(r[(1,3)], tau12 + tau21))

R[(2,3)] = 0
R[(3,2)] = 0

In [51]:
a = {
    (1,2): omega1 - omega2,
    (2,1): omega2 - omega1,
    (1,3): omega1 - omega3,
    (3,1): omega3 - omega1,
    (2,3): omega2 - omega3,
    (3,2): omega3 - omega2,
}

# clockwise triplets I+_3
triplets = [(1,2,3), (2,3,1), (3,1,2)]

correction = 0
for (i, j, k) in triplets:
    correction -= (
        - a[(i,j)] * R[(i,j)]
        - a[(i,k)] * R[(i,k)]
    )

X1c = simplify(X1 + correction)

In [52]:
print("The X1 expression is:")
display(Math(r'X_1 = ' + color_terms(latex(X1))))
#print("The clock-jitter correction term is:")
#display(Math(r'X_1 = ' + color_terms(latex(simplify(-correction)))))
print("The corrected X1 expression is:")
display(Math(r'X_1 = ' + color_terms(latex(X1c))))

The X1 expression is:


<IPython.core.display.Math object>

The corrected X1 expression is:


<IPython.core.display.Math object>

In [53]:
REF_AB= REF[(1,3)] - REF[(2,3)]
REF_AC= REF[(3,2)] - REF[(1,2)]
REF_CB= REF[(2,1)] - REF[(3,1)]

display(Math(r'REF_{AB} = ' + color_terms(latex(REF_AB))))
display(Math(r'REF_{AC} = ' + color_terms(latex(REF_AC))))
display(Math(r'REF_{CB} = ' + color_terms(latex(REF_CB))))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [54]:
Corr1 = omega1*(D(REF_AB, tau12) - D(REF_AB, tau12 + tau21) - D(REF_AB, tau12 + tau13 + tau31))
Corr2 = omega1*(D(REF_AC, tau13) - D(REF_AC, tau13 + tau31) - D(REF_AC, tau12 + tau13 + tau21))
Corr3 = -omega1*(D(REF_CB, tau12 + tau13 + tau21 + tau31))
display(Math(r'X_1 = ' + color_terms(latex(simplify(X1c + Corr1 + Corr2 + Corr3)))))

<IPython.core.display.Math object>

Let's try with Sagnac

In [55]:
# Sagnac alpha_1 centered on SC1
# alpha_1 = eta12 + D12*eta23 + D123*eta31 - eta13 - D13*eta32 - D132*eta21

def P12_sagnac(expr): return expr
def P23_sagnac(expr): return D(expr, tau12)
def P31_sagnac(expr): return D(expr, tau12 + tau23)
def P13_sagnac(expr): return -expr
def P32_sagnac(expr): return -D(expr, tau13)
def P21_sagnac(expr): return -D(expr, tau13 + tau32)

R_sagnac = {}
R_sagnac[(1,2)] = 0
R_sagnac[(2,3)] = r[(1,2)]
R_sagnac[(3,1)] = r[(1,2)] + D(r[(2,3)], tau12)
R_sagnac[(1,3)] = 0
R_sagnac[(3,2)] = -r[(1,3)]
R_sagnac[(2,1)] = -(r[(1,3)] + D(r[(3,2)], tau13))

In [56]:
a = {
    (1,2): omega1 - omega2,
    (2,1): omega2 - omega1,
    (1,3): omega1 - omega3,
    (3,1): omega3 - omega1,
    (2,3): omega2 - omega3,
    (3,2): omega3 - omega2,
}

# clockwise triplets I+_3
triplets = [(1,2,3), (2,3,1), (3,1,2)]

correction = 0
for (i, j, k) in triplets:
    correction -= (
        - a[(i,j)] * R_sagnac[(i,j)]
        - a[(i,k)] * R_sagnac[(i,k)]
    )

alpha1 = collect(expand(
      P12_sagnac(eta[(1,2)])
    + P23_sagnac(eta[(2,3)])
    + P31_sagnac(eta[(3,1)])
    + P13_sagnac(eta[(1,3)])
    + P32_sagnac(eta[(3,2)])
    + P21_sagnac(eta[(2,1)])
), [q1(t), q2(t), q3(t), epsilonA(t), epsilonB(t), epsilonC(t)])


alpha1c = simplify(alpha1 + correction)

print("The alpha_1 expression is:")
display(Math(r'\alpha_1 = ' + color_terms(latex(alpha1))))  
print("The corrected alpha_1 expression is:")
display(Math(r'\alpha_1 = ' + color_terms(latex(alpha1c))))

The alpha_1 expression is:


<IPython.core.display.Math object>

The corrected alpha_1 expression is:


<IPython.core.display.Math object>

In [57]:
CorrA_sagnac = omega1 * (D(REF_AC, tau13) + D(REF_AB, tau12))
CorrBC_sagnac = omega1 * (D(REF_CB, tau12+tau23) + D(REF_CB, tau13+tau32) - D(REF_CB, tau12 + tau23 + tau31))

alpha1_corrected = simplify(alpha1c + CorrA_sagnac + CorrBC_sagnac).subs({tau12:tau21, tau13: tau31, tau23: tau32})
print("The alpha_1 expression with only REF corrections is:")
display(Math(r'\alpha_1 = ' + color_terms(latex(alpha1_corrected))))

The alpha_1 expression with only REF corrections is:


<IPython.core.display.Math object>

Let's see if we can get rid of the board jitters before the TDI

cant